# Lecture 8: Structure Optimisation

## Overview
**Questions**
- How do I find the minimum energy structure?
- How do I apply constraints during optimisation?
- How do I optimise a defect geometry?

**Objectives**
- Use BFGS and FIRE optimisers to relax a structure
- Apply constraints (fix certain atoms)
- Understand the importance of geometry relaxation for defect calculations


## Lecture Slides

The slides for this tutorial are embedded below.
[📥 Download slides (.pptx)](https://github.com/NU-CEM/Atomistic_Simulation/raw/2026/slides/Tutorial6_LocalOptimisation.pptx) &nbsp;|&nbsp; [Open in full screen](SHAREPOINT_EMBED_URL_HERE)

<iframe
  src="SHAREPOINT_EMBED_URL_HERE"
  width="100%"
  height="480"
  frameborder="0"
  allowfullscreen="true">
</iframe>

> **How to embed:** Upload the .pptx to PowerPoint Online (SharePoint/OneDrive) → File → Share → Embed → copy the `src="..."` URL and replace `SHAREPOINT_EMBED_URL_HERE` above.
> The download link already points to `slides/Tutorial6_LocalOptimisation.pptx` on GitHub — just commit the file to that path.

---

## Why Geometry Optimisation?

Real materials are not perfectly periodic: defects distort the surrounding lattice, surfaces reconstruct, and molecules adopt their lowest-energy conformation. Before computing any electronic property, we must find the minimum energy geometry.

For quantum optics:
- The NV⁻ centre in diamond has **C₃ᵥ symmetry** after relaxation — the N and three C neighbours around the vacancy all move inward
- The emission energy of a defect depends sensitively on the relaxed geometry of both ground and excited states
- Computing the **Huang-Rhys factor** requires the geometry change between ground and excited states


In [ ]:
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.optimize import BFGS, FIRE
from ase import units
import numpy as np
import matplotlib.pyplot as plt

# Build a deliberately strained Al structure
al = bulk('Al', 'fcc', a=4.20)   # a0 = 4.05 Å — this is stretched
al.calc = EMT()

print(f"Initial energy: {al.get_potential_energy():.6f} eV")
print(f"Initial max force: {np.max(np.abs(al.get_forces())):.6f} eV/Å")

# Optimise with BFGS
opt = BFGS(al, logfile='/tmp/al_bfgs.log')
opt.run(fmax=0.001)

print(f"\nFinal energy: {al.get_potential_energy():.6f} eV")
print(f"Final max force: {np.max(np.abs(al.get_forces())):.6f} eV/Å")
print(f"Final lattice constant: {al.cell[0,0]:.4f} Å (expected ~4.05 Å for EMT)")


In [ ]:
# Read the log file and plot convergence
import re

steps, fmax_vals = [], []
with open('/tmp/al_bfgs.log') as f:
    for line in f:
        m = re.match(r'\s+(\d+)\s+[\d.]+\s+([\d.]+)', line)
        if m:
            steps.append(int(m.group(1)))
            fmax_vals.append(float(m.group(2)))

if steps:
    plt.figure(figsize=(7, 4))
    plt.semilogy(steps, fmax_vals, 'o-', color='steelblue')
    plt.axhline(0.001, color='red', ls='--', label='fmax threshold = 0.001 eV/Å')
    plt.xlabel('Optimisation step')
    plt.ylabel('Max force (eV/Å)')
    plt.title('BFGS geometry optimisation convergence')
    plt.legend()
    plt.tight_layout()
    plt.show()


## Optimising with Constraints

For surface or defect calculations, we often want to **fix** some atoms (e.g. the bottom layers of a slab) while relaxing others. ASE's `ase.constraints` module provides this.


In [ ]:
from ase.build import surface
from ase.constraints import FixAtoms

# Build a GaN-like surface (using Al as proxy for EMT demo)
al_slab = bulk('Al', 'fcc', a=4.05)
from ase.build import fcc111
al_surface = fcc111('Al', size=(3, 3, 5), vacuum=10.0)
al_surface.calc = EMT()

print(f"Surface slab: {len(al_surface)} atoms")
print(f"Z coordinates: {np.unique(al_surface.positions[:,2].round(2))}")

# Fix bottom 2 layers (lowest z positions)
z_coords = al_surface.positions[:,2]
z_threshold = np.sort(np.unique(z_coords.round(2)))[1]  # 2nd unique z layer
fixed_mask = z_coords <= z_threshold + 0.1
fix = FixAtoms(mask=fixed_mask)
al_surface.set_constraint(fix)

n_fixed = sum(fixed_mask)
print(f"Fixed atoms: {n_fixed} / {len(al_surface)}")

# Optimise the surface
opt_surf = BFGS(al_surface, logfile='/dev/null')
opt_surf.run(fmax=0.05)
print(f"Surface relaxation complete. Max force: {np.max(np.abs(al_surface.get_forces())):.4f} eV/Å")


## Trajectories

A **trajectory** is a sequence of `Atoms` objects, typically snapshots from a molecular dynamics run or a geometry optimisation. ASE has a native `.traj` format for storing these efficiently.


In [ ]:
from ase.io.trajectory import Trajectory
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.optimize import BFGS

# Quick example: optimise Al and record the trajectory
al = bulk('Al', 'fcc', a=4.10)   # slightly off equilibrium
al.calc = EMT()

traj_path = '/tmp/al_opt.traj'
opt = BFGS(al, trajectory=traj_path, logfile='/tmp/al_opt.log')
opt.run(fmax=0.01)

# Read the trajectory back
from ase.io import read as ase_read
traj = ase_read(traj_path, index=':')  # ':' means all frames
print(f"Optimisation took {len(traj)} steps")
print(f"Initial energy:  {traj[0].get_potential_energy():.4f} eV")
print(f"Final energy:    {traj[-1].get_potential_energy():.4f} eV")

In [ ]:
# Plot the energy convergence during optimisation
import matplotlib.pyplot as plt

energies = [atoms.get_potential_energy() for atoms in traj]
plt.figure(figsize=(7, 4))
plt.plot(energies, 'o-', color='steelblue')
plt.xlabel('Optimisation step')
plt.ylabel('Energy (eV)')
plt.title('Al geometry optimisation convergence')
plt.tight_layout()
plt.show()


## Defect Geometry Relaxation

The most important use case for geometry optimisation in quantum optics is **relaxing a defect supercell**. After introducing the defect atoms, the surrounding lattice must relax to accommodate the perturbation.

Here is the full workflow (using EMT as proxy for DFT):


In [ ]:
from ase.build import bulk, make_supercell
from ase.constraints import FixSymmetry

# 1. Build a 3×3×3 diamond supercell
diamond = bulk('C', 'diamond', a=3.57)
T = np.diag([3, 3, 3])
nv_cell = make_supercell(diamond, T)

# 2. Create NV-like defect (N substitution + vacancy)
from ase.geometry import get_distances
_, dist_matrix = get_distances(nv_cell.positions, nv_cell.positions,
                                cell=nv_cell.cell, pbc=True)
np.fill_diagonal(dist_matrix, np.inf)

# Substitute atom 0 → N, delete nearest C
symbols = list(nv_cell.get_chemical_symbols())
symbols[0] = 'N'
nv_cell.set_chemical_symbols(symbols)
vacancy_idx = np.argmin(dist_matrix[0])
del nv_cell[vacancy_idx]

print(f"NV supercell: {nv_cell.get_chemical_formula()}, {len(nv_cell)} atoms")

# 3. Use EMT as a fast proxy calculator
# (In real research: use GPAW or QE with PBE/HSE06)
nv_cell.calc = EMT()

# 4. Optimise
opt_nv = BFGS(nv_cell, logfile='/dev/null')
opt_nv.run(fmax=0.05)

print(f"Relaxation complete.")
print(f"Max residual force: {np.max(np.abs(nv_cell.get_forces())):.4f} eV/Å")


## Key Points

- Geometry optimisation finds the minimum energy atomic configuration
- **BFGS** is the standard choice; **FIRE** can be better for large systems
- `FixAtoms` constrains selected atoms during optimisation
- Convergence criterion `fmax` is the maximum force component (typically 0.01–0.05 eV/Å for DFT)
- Defect supercell relaxation is **essential** before computing electronic or optical properties

## Exercise 8.1

Build a 2×2×2 supercell of wurtzite GaN and introduce a nitrogen vacancy (V_N) by deleting one N atom. Relax the structure with BFGS using EMT. By how much does the nearest Ga atom move inward toward the vacancy? (This "breathing mode" relaxation is a characteristic signature of point defects.)

## Exercise 8.2

Look up the concept of the **configuration coordinate diagram** (CCD). How is it related to geometry optimisation in the ground and excited states? Sketch a CCD for the NV centre, labelling the zero-phonon line (ZPL) energy, the Stokes shift, and the reorganisation energy.


## Exercise 6.3 — Coursework Preparation: Relaxing Your Defect Structure

1. Load your `coursework_defect_initial.xyz` from Tutorial 4.
2. Attach the MACE-MP-0 calculator (from Tutorial 8) and relax the structure with BFGS 
   (fmax = 0.01 eV/Å).
3. Plot the convergence of the maximum force vs optimisation step.
4. Save the relaxed structure as `coursework_defect_relaxed.xyz`.

Answer these questions in a markdown cell:
- How many steps did relaxation take? Is this more or less than you expected?
- By how much did the atoms nearest the defect move (in Å)?  
- Does the relaxed structure preserve the expected point symmetry of the defect?

This relaxed geometry is the starting point for your DFT electronic structure calculation.

*Note: If MACE is not yet installed, use EMT as a placeholder and revisit after Tutorial 8.*
